# UKRI analysis

In [559]:
import pandas as pd
from discovery_child_development import PROJECT_DIR

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [560]:
# AltairSaver = altair_save_utils.AltairSaver()

In [561]:
import utils
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import importlib
importlib.reload(utils);

## Load data

In [565]:
# Gateway to Research labelled data
data_df = utils.load_ukri_data().query("topics != 'arts'")

In [564]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [567]:
# Transform to one id and topic pair per row
importlib.reload(utils)
data_exploded_df = utils.explode_data(data_df).query("topics != 'arts'")

## Baseline trends

Baseline UKRI trends for funding and project counts 

In [568]:
baseline_df = utils.get_baseline_ukri()

In [569]:
baseline_df

,year,counts,amount
0,2013,6958,2946558.257
1,2014,6612,3412503.237
2,2015,8187,3155833.595
3,2016,7949,3179630.580
4,2017,10623,3793365.379
5,2018,9933,4994989.471
6,2019,10814,4157180.133
7,2020,13451,4125481.567
8,2021,11131,3648542.132
9,2022,12192,4490129.906


In [570]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,1.098500e+04,-2.263309
amount,4.114764e+06,-5.054841


In [571]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [572]:
ts_counts = utils.get_timeseries(data_df, column='id')
ts_amounts = utils.get_timeseries(data_df, column='amount')

In [573]:
ts_amounts

,year,amount
0,2013,37095.359
1,2014,66541.735
2,2015,28813.706
3,2016,31994.220
4,2017,35872.551
5,2018,51963.333
6,2019,60558.017
7,2020,55885.492
8,2021,62901.780
9,2022,54392.334


In [574]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,112.6,-18.4573


In [575]:
au.ts_magnitude_growth_(
    ts_df = ts_amounts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
amount,58533.9486,18.755712


In [576]:
fig = pu.ts_smooth(
    ts_amounts.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

Funding for the overall early-years development related research has increased by about 19% in the past five years, which is a positive trend compared to baseline funding which slightly decreased by about 5% in the same time period.

In [577]:
utils.get_data_distribution(data_exploded_df, column='type', values=['id', 'amount'])

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,256,0.236,167868.383,0.308
1,Child care & preschool,40,0.037,27351.956,0.05
2,Development & learning,246,0.227,95427.613,0.175
3,General,673,0.62,357276.138,0.656
4,Health,675,0.622,385818.684,0.708
5,Parenting,36,0.033,22139.205,0.041
6,Social,233,0.215,114638.455,0.21
7,Technology,168,0.155,82609.868,0.152


In [601]:
importlib.reload(utils)
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id', 'amount'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='amount')

,magnitude,growth,type,counts
7,11767.7818,165.741503,Technology,105
5,1624.2724,92.789354,Parenting,20
4,41972.1448,17.383139,Health,362
3,39058.3124,11.895499,General,348
2,11291.2502,5.226451,Development & learning,123
0,16878.7816,-5.314112,Biosciences,125
6,13691.3442,-8.089196,Social,133
1,3218.7396,-73.870816,Child care & preschool,23


In [579]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "amount",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [584]:
## Check inflation
importlib.reload(utils)
ts_adjusted = utils.adjust_by_uk_inflation(ts_amounts)
ts_baseline_adjusted = utils.adjust_by_uk_inflation(baseline_df)
au.ts_magnitude_growth_(
    ts_df = ts_adjusted,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
amount,58533.948600,18.755712
real_terms,54538.000957,5.369052


In [585]:
au.ts_magnitude_growth_(
    ts_df = ts_baseline_adjusted,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,1.098500e+04,-2.263309
amount,4.114764e+06,-5.054841
real_terms,3.826380e+06,-16.272233


## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [586]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [587]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [594]:
ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [589]:
au.ts_magnitude_growth_(ts_amounts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
amount,11767.7818,165.741503


### Distribution of different technologies

In [590]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [591]:
# Total tech funding
amount_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").amount.sum()

In [592]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
    .assign(amount_prop = lambda df: round(df.amount / amount_total, 3))
)

tech_subtype_dist

,subtype,counts,amount,amount_prop
0,AI,62,40910.811,0.695
1,Immersive tech,24,7718.759,0.131
2,Internet,18,5814.574,0.099
3,Mobile,23,17869.267,0.304


### Growth of technology topics

In [596]:
column = 'subtype'
value = 'amount'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,8182.1622,147.968645,AI
0,1543.7518,23.475780,Immersive tech
0,1162.9148,378.897783,Internet
0,3573.8534,87.113273,Mobile


In [597]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [598]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [599]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)


In [600]:
tech_applications_df

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,23,0.219,13298.835,0.226
1,Child care & preschool,8,0.076,2590.215,0.044
2,Development & learning,31,0.295,12640.91,0.215
3,General,61,0.581,32165.322,0.547
4,Health,64,0.61,40250.779,0.684
5,Parenting,9,0.086,3059.329,0.052
6,Social,21,0.2,16255.875,0.276
7,Technology,105,1.0,58838.909,1.0


In [527]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [528]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount')

,magnitude,growth,type,counts
6,611.8658,2541.467658,Parenting,9
4,518.0430,299.955036,Child care & preschool,8
1,6433.0644,245.062823,General,61
2,8050.1558,210.796765,Health,64
3,11767.7818,165.741503,Technology,105
7,3251.1750,148.675879,Social,21
0,2659.7670,65.806908,Biosciences,23
5,2670.9104,-43.751353,Development & learning,33


In [531]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )

### Application distribution: More granular subtypes

In [602]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('amount', ascending=False)

,subtype,counts,counts_prop,amount,amount_prop,type
7,Health,45,0.429,26284.452,0.447,Health
12,Infancy,49,0.467,24237.342,0.412,General
26,Prenatal,22,0.21,13180.068,0.224,Health
16,Mental health,15,0.143,12150.008,0.206,Health
11,Inequalities,10,0.095,11243.437,0.191,Social
31,Special educational needs,19,0.181,9652.065,0.164,Development & learning
20,Nutrition & weight,12,0.114,8494.048,0.144,Health
18,Neuroscience,18,0.171,7773.964,0.132,Biosciences
6,Genetics,10,0.095,7545.618,0.128,Biosciences
30,Social services,11,0.105,7287.046,0.124,Social


In [604]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount').sort_values(['type', 'growth'], ascending=False)

,magnitude,growth,subtype,counts,type
11,1162.9148,378.897783,Internet,18,Technology
14,8182.1622,147.968645,AI,62,Technology
15,3573.8534,87.113273,Mobile,23,Technology
21,1543.7518,23.475780,Immersive tech,24,Technology
0,414.8748,inf,Inclusion,2,Social
4,592.8278,inf,Community,7,Social
16,113.2234,80.543428,Income,3,Social
18,2248.6874,60.622139,Inequalities,10,Social
24,1457.4092,-42.078372,Social services,11,Social
31,14.8326,-100.000000,Labour market,2,Social


In [534]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [535]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)